# Paper Fig. 2: gas-drag baseline

This notebook implements the **10 km aerodynamic drag baseline** from Zheng, Lin & Kouwenhoven (2016), equations (1), (2), (5), and (6). It deliberately omits Jupiter, Saturn, disk gravity, mean-motion resonances, and sweeping secular resonances. Its tracks therefore **are not a reproduction of Fig. 2**. The purpose is to validate local disk/drag physics before adding the missing gravitational dynamics.

The paper disk and the existing `accrediff.gas_migration` disk are accessed through the same `DiskProfile` interface. Gas-flow normalization and drag-law normalization are configured separately.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src" / "accrediff").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("AccreDiff repository root not found")
HERE = PROJECT_ROOT / "examples" / "Expansion_module" / "GasDrag_Type1"
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from accrediff.gas_migration import Gas_ModelConfig, Gas_build_model
from paper_fig2_disk import PaperFig2Disk, GasMigrationDiskAdapter
from paper_fig2_drag import Planetesimal, GasFlowPolicy, AerodynamicDrag
from paper_fig2_orbit_average import orbit_averaged_rates

## Configurable baseline

The paper uses a 10 km **radius**, internal density 1 g/cm³, `C_D=0.165`, `T_dep=1 Myr`, and start radii 3.5, 3.2, 2.7, and 2.3 AU. Initial eccentricity is an illustrative baseline here; the paper's resonant excitation is not modeled.

In [ ]:
paper_disk = PaperFig2Disk()
body = Planetesimal(radius_km=10.0, solid_density_cgs=1.0)
flow = GasFlowPolicy(convention="paper_2016")
drag = AerodynamicDrag(body, flow, law="paper_2016", drag_coefficient=0.165)
initial_a_au = (3.5, 3.2, 2.7, 2.3)
e0 = 0.01  # illustrative, not a reconstructed resonant initial condition
t_end_myr = 10.0
n_orbit_samples = 128
print(f"10 km body: mass = {body.mass_g:.6e} g = {body.mass_mearth:.6e} Mearth")
print(f"Paper disk at 1 AU, t=0: Sigma = {paper_disk.sigma_cgs(1,0):g} g/cm², "
      f"H = {paper_disk.scale_height_au(1,0):g} AU")

## Disk adapter check

The first existing-disk configuration is algebraically equivalent to the paper's un-gapped Σ and H. The second retains the `gas_profile.ipynb` radial profile, making the effect of changing *only the disk structure* visible. All three use the same paper gas-flow and drag prescriptions below.

In [ ]:
matched_cfg = Gas_ModelConfig(Sigma0_cgs=1700, a_alpha1=0,
                              a_alpha2=1.5, def_h_1=0.025, beta=0.25,
                              tau_decay_myr=1.0)
matched_model = Gas_build_model(matched_cfg)
matched_disk = GasMigrationDiskAdapter(matched_model["disk"], matched_model["units"])
existing_cfg = Gas_ModelConfig(Sigma0_cgs=1700, a_alpha1=1.0,
                               a_alpha2=0.5, tau_decay_myr=1.0)
existing_model = Gas_build_model(existing_cfg)
existing_disk = GasMigrationDiskAdapter(existing_model["disk"], existing_model["units"])

r_plot = np.linspace(0.5, 4.0, 400)
assert np.allclose(paper_disk.sigma_cgs(r_plot,0), matched_disk.sigma_cgs(r_plot,0))
assert np.allclose(paper_disk.scale_height_au(r_plot,0), matched_disk.scale_height_au(r_plot,0))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for label, d, ls in [("paper", paper_disk, "-"),
                     ("existing adapter, matched", matched_disk, "--"),
                     ("existing adapter, gas_profile defaults", existing_disk, ":")]:
    axes[0].plot(r_plot, d.sigma_cgs(r_plot, 0), ls=ls, label=label)
    axes[1].plot(r_plot, d.scale_height_au(r_plot, 0), ls=ls, label=label)
axes[0].set(xlabel="Radius (AU)", ylabel=r"$\Sigma_g$ (g cm$^{-2}$)", yscale="log")
axes[1].set(xlabel="Radius (AU)", ylabel="Scale height H (AU)")
for ax in axes: ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
fig.suptitle("Interchangeable gas-disk profiles at t=0")
fig.tight_layout()
plt.show()

## Drag diagnostics

The table gives instantaneous orbit-averaged drag rates at `t=0`. The quoted `a/|da/dt|` is not a traveled distance and does not include resonant forcing.

In [ ]:
print("a0 (AU)   Sigma (g/cm²)  rho_mid (g/cm³)  eta       da/dt (AU/Myr)  de/dt (1/Myr)  a/|da/dt| (yr)")
for a0 in initial_a_au:
    rates = orbit_averaged_rates(a0, e0, 0.0, paper_disk, drag,
                                 n_samples=n_orbit_samples)
    timescale = a0 / abs(rates.da_dt_au_per_myr) * 1e6
    print(f"{a0:7.1f} {paper_disk.sigma_cgs(a0,0):16.5f} "
          f"{flow.density_cgs(paper_disk,a0,0):16.4e} "
          f"{flow.eta(paper_disk,a0,0):8.3e} "
          f"{rates.da_dt_au_per_myr:16.6e} "
          f"{rates.de_dt_per_myr:14.6e} {timescale:16.4e}")

## Drag-only secular tracks

These are deliberately smooth: without Jupiter/Saturn and the decaying disk's gravitational precession, there are no MMR or ν5/ν6 kicks. The integration stops at 0.05 AU if needed. `q` and `Q` are shown to establish the plotting pipeline for a later full Fig. 2 calculation.

In [ ]:
t_plot = np.linspace(0, t_end_myr, 400)

def integrate_drag_only(a0_au, disk_profile=paper_disk):
    def rhs(t, state):
        a, e = state
        rates = orbit_averaged_rates(max(a, 0.05), np.clip(e, 0, 0.999),
                                     t, disk_profile, drag, n_orbit_samples)
        return rates.da_dt_au_per_myr, rates.de_dt_per_myr
    def inner_edge(t, state):
        return state[0] - 0.05
    inner_edge.terminal = True
    inner_edge.direction = -1
    sol = solve_ivp(rhs, (0, t_end_myr), (a0_au, e0), dense_output=True,
                    events=inner_edge, rtol=1e-7, atol=1e-10, max_step=0.05)
    if not sol.success:
        raise RuntimeError(sol.message)
    a = np.full_like(t_plot, np.nan)
    e = np.full_like(t_plot, np.nan)
    good = t_plot <= sol.t[-1] + 1e-12
    a[good], e[good] = sol.sol(t_plot[good])
    e[good] = np.clip(e[good], 0, 0.999)
    return a, e

tracks = {a0: integrate_drag_only(a0) for a0 in initial_a_au}
fig, ax = plt.subplots(figsize=(8, 4.8))
for color, a0 in zip(("tab:red", "tab:green", "tab:blue", "tab:purple"), initial_a_au):
    a, e = tracks[a0]
    ax.plot(t_plot, a, color=color, lw=2, label=f"a0={a0:g} AU")
    ax.plot(t_plot, a*(1-e), color=color, lw=0.9, alpha=0.7)
    ax.plot(t_plot, a*(1+e), color=color, lw=0.9, alpha=0.7)
ax.set(xlabel="Time (Myr)", ylabel="a, q, Q (AU)",
       title="Drag-only baseline: no planetary or secular resonances")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()
for a0 in initial_a_au:
    a, e = tracks[a0]
    last = a[np.isfinite(a)][-1]
    print(f"a0={a0:g} AU -> a(10 Myr)={last:.6f} AU, inward displacement={a0-last:.6f} AU")

## Interpretation and sources

The smooth drag-only result is a validation baseline, not Fig. 2. A faithful Fig. 2 needs Jupiter/Saturn gravity, mean-motion resonances, and disk-gravity-driven ν5/ν6 sweeping. The paper's local drag coefficient is not the same normalization as `gas.cu` for an equal numerical `C_D`.

Sources: Zheng, Lin & Kouwenhoven (2016), arXiv:1610.09670v1, equations (1), (2), (5), (6) and Fig. 2; Adachi, Hayashi & Nakazawa (1976), *Progress of Theoretical Physics* 56, 1756–1771, DOI 10.1143/PTP.56.1756.